# Experimental MCP Workflow Walkthrough

This generated notebook is the recipe companion for
`examples/mcp_workflow_demo.py`.

Launch the local stdio MCP server, inspect its public contract, preview a production logfile, exercise the writable authoring tools, and walk a copied header packet through deterministic MCP ingestion without dropping down to the direct Python or YAML pipeline.

What you will practice in this walkthrough:

- Start the experimental `wellplot-mcp` surface from a notebook-friendly Python workflow.
- Inspect the registered tools, resources, resource templates, and prompts before using them.
- Use MCP tool calls to validate, inspect, and preview a production logfile at full-report, section, track, and window scopes.
- Round-trip a full logfile through MCP text validation, formatting, export, save, and explicit render-to-file calls.
- Parse copied header text, inspect exact heading slots, preview the mapping, apply it, and preview the first report page through MCP.

Prerequisites:

- `pip install "wellplot[dlis,mcp,notebook]"`
- run the notebook from a checkout of this repository so the `examples/` files and sample data are available

Runtime model:

- import `wellplot` from the active installed environment
- use the repository checkout for the example files, helper modules, and sample data


In [ ]:
import sys
from pathlib import Path

try:
    import wellplot
except ImportError as exc:
    raise RuntimeError(
        "Install the published 'wellplot' package in the active "
        "environment before running this notebook."
    ) from exc

# Walk upward from the current working directory until we find the
# repository checkout that holds the example sources and sample data.
cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "examples").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "Run this notebook from a checkout of the wellplot repository "
        "so the example files and sample data are available."
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

examples_path = str(EXAMPLES_DIR)
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

print("wellplot version:", wellplot.__version__)
print("Examples root:", EXAMPLES_DIR)
print("Render output:", WORKSPACE_RENDERS)

In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "mcp_workflow_demo.py"
display(Code(source_path.read_text(), language="python"))

In [ ]:
# Import the MCP helper module and inspect the default example paths it
# uses for the walkthrough.
import mcp_workflow_demo as demo

print("Demo logfile:", demo.DEFAULT_LOGFILE)
print("Header-ingestion logfile:", demo.DEFAULT_HEADER_LOGFILE)
print("Demo base dir:", demo.DEFAULT_BASE_DIR)
print("Demo output root:", demo.DEFAULT_OUTPUT_ROOT)

In [ ]:
# Ask the MCP server what it exposes before calling the workflow tools.
contract = await demo.collect_server_contract()

print("Tools:")
for name in contract["tools"]:
    print(" -", name)

print("\nResources:")
for uri in contract["resources"]:
    print(" -", uri)

print("\nResource templates:")
for uri_template in contract["resource_templates"]:
    print(" -", uri_template)

print("\nPrompts:")
for name in contract["prompts"]:
    print(" -", name)

print("\nPackaged production examples:")
for example in contract["example_manifest"]["examples"]:
    print(f" - {example['id']}: {example['title']}")

print("\nreview_logfile prompt preview:\n")
print(contract["review_prompt"])

In [ ]:
# Validate, inspect, and preview the production logfile through MCP only.
from IPython.display import Image, display

review = await demo.run_review_flow()

print("Validation:")
print(review["validation"])
print("\nSelected section:", review["selected_section_id"])
print("Selected tracks:", review["selected_track_ids"])
print("Window depth range:", review["window_depth_range"])
print("\nResolved section ids:", review["inspection"]["section_ids"])

display(Image(data=review["section_preview_png"]))
display(Image(data=review["track_preview_png"]))
display(Image(data=review["window_preview_png"]))

In [ ]:
# Exercise the writable MCP tools against the same production example.
from IPython.display import Code, display

authoring = await demo.run_authoring_flow()

print("Text validation:")
print(authoring["text_validation"])
print("\nExported files:")
for path in authoring["export"]["written_files"]:
    print(" -", path)
print("\nSaved logfile:", authoring["save"]["output_path"])
print("Rendered logfile:", authoring["rendered_logfile"])
print("Rendered PDF:", authoring["render"]["output_path"])

preview_yaml = "\n".join(authoring["formatted_yaml"].splitlines()[:80])
display(Code(preview_yaml, language="yaml"))

In [ ]:
# Clone a draft, parse copied header text, preview the mapping, apply it,
# and preview the first report page through MCP only.
import json
from IPython.display import Image, Code, display

header_demo = await demo.run_header_ingestion_flow()

print("Header draft:", header_demo["draft_logfile"])
print("Mapped values:")
print(header_demo["mapped_values"])

print("\nParsed pairs:")
for pair in header_demo["parsed"]["pairs"]:
    print(f" - {pair['key']}: {pair['value']}")

print("\nPrompt excerpt:\n")
print("\n".join(header_demo["ingest_prompt"].splitlines()[:14]))

print("\nReport-page style presets:")
for preset in header_demo["style_presets"]["presets"]:
    print(f" - {preset['id']}: {preset['label']}")

print("\nPreview warnings:", header_demo["preview"]["warnings"])
print("Applied assignments:", len(header_demo["apply"]["applied_assignments"]))

predicted_patch = json.dumps(
    header_demo["preview"]["predicted_heading_patch"],
    indent=2,
)
display(Code(predicted_patch, language="json"))
display(Image(data=header_demo["page_preview_png"]))

## Adapt Experimental MCP Workflow Walkthrough Safely

- Keep the server root pointed at a narrow working directory when you move from a repo demo to a real job directory.
- Use `inspect_logfile(...)` first and then drive the narrow preview tools with the returned section and track ids.
- Treat `format_logfile_text(...)` and `save_logfile_text(...)` as normalization steps, not as comment-preserving editors.
- Use explicit keys like `general_field.company`, `detail.date`, or `service_title_1` when copied header text could map to multiple slots.
